In [1]:
import re
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
from datasets import load_dataset

dataset = load_dataset("Wangrongsheng/ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

train_texts = train_data["text"]
train_labels = train_data["label"]

test_texts = test_data["text"]
test_labels = test_data["label"]

print("Training Samples:", len(train_texts))
print("Testing Samples:", len(test_texts))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training Samples: 120000
Testing Samples: 7600


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

train_texts = [clean_text(text) for text in train_texts]
test_texts = [clean_text(text) for text in test_texts]

In [4]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)


In [5]:
max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [6]:
y_train = to_categorical(train_labels, num_classes=4)
y_test = to_categorical(test_labels, num_classes=4)


In [7]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_length
    )
)

model.add(SimpleRNN(64))

model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.7669 - loss: 0.6505 - val_accuracy: 0.8345 - val_loss: 0.4866
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8858 - loss: 0.3577 - val_accuracy: 0.8786 - val_loss: 0.3621
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9099 - loss: 0.2856 - val_accuracy: 0.8793 - val_loss: 0.3678
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9206 - loss: 0.2495 - val_accuracy: 0.8858 - val_loss: 0.3547
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9328 - loss: 0.2123 - val_accuracy: 0.8753 - val_loss: 0.3968


In [10]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8845 - loss: 0.3599

Test Loss: 0.35990044474601746
Test Accuracy: 0.8844736814498901


In [12]:
labels = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

sample_news = [
    "Apple launches a new AI-powered iPhone with advanced features."
]

sample_news = [clean_text(text) for text in sample_news]

sequence = tokenizer.texts_to_sequences(sample_news)

sequence = pad_sequences(
    sequence,
    maxlen=max_length,
    padding='post'
)

In [13]:
prediction = model.predict(sequence)

predicted_class = np.argmax(prediction)

print("\nNews:", sample_news[0])
print("Predicted Category:", labels[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 520ms/step

News: apple launches a new aipowered iphone with advanced features
Predicted Category: Science/Technology
